# OF `chisq_mode='none'` Smoke Test

Quick validation for `reusable/OF_tutorial/OptimumFilter.py` after the `chisq_mode='none'` fix.

In [1]:
from __future__ import annotations

import importlib
import json
import sys
from pathlib import Path

import numpy as np

BASE_DIR = Path.cwd()
CONFIG_PATH = BASE_DIR / 'of_smoke_config.json'
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

default_cfg = {
    'sampling_frequency': 250000.0,
    'uv_template_path': '/home/dwong/DELight_mtr/trigger_study/archive/wk15/templates/vac_ch_template.npy',
    'noise_psd_path': '/home/dwong/DELight_mtr/trigger_study/archive/wk29/pink_psd.npy',
    'seed': 123,
    'long_trace_noise_std': 0.03,
    'hop': 16,
}

if CONFIG_PATH.exists():
    cfg = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
    print(f'Loaded config from {CONFIG_PATH}')
else:
    cfg = default_cfg
    print('Using built-in fallback config (of_smoke_config.json not found).')

cfg

Using built-in fallback config (of_smoke_config.json not found).


{'sampling_frequency': 250000.0,
 'uv_template_path': '/home/dwong/DELight_mtr/trigger_study/archive/wk15/templates/vac_ch_template.npy',
 'noise_psd_path': '/home/dwong/DELight_mtr/trigger_study/archive/wk29/pink_psd.npy',
 'seed': 123,
 'long_trace_noise_std': 0.03,
 'hop': 16}

In [2]:
template_path = Path(cfg['uv_template_path'])
noise_psd_path = Path(cfg['noise_psd_path'])
fs = float(cfg['sampling_frequency'])

template = np.load(template_path).astype(np.float64)
noise_psd = np.load(noise_psd_path).astype(np.float64)

old_mod = importlib.reload(importlib.import_module('OptimumFilter_old'))
new_mod = importlib.reload(importlib.import_module('OptimumFilter'))
of_old = old_mod.OptimumFilter(template, noise_psd, fs)
of_new = new_mod.OptimumFilter(template, noise_psd, fs)

print('template length:', template.size)
print('noise bins:', noise_psd.size)
print('fs:', fs)

template length: 32768
noise bins: 16385
fs: 250000.0


In [3]:
rng = np.random.default_rng(int(cfg.get('seed', 123)) + 10)
N = template.size
L = N + 4096
trace_long = rng.normal(0.0, float(cfg.get('long_trace_noise_std', 0.03)), size=L)
for s, a in [(0, 1.1), (1024, 0.9), (2048, 1.3), (3072, 0.7)]:
    if s + N <= L:
        trace_long[s:s + N] += a * template

print('trace_long shape:', trace_long.shape)

trace_long shape: (36864,)


In [4]:
# hop=1: check old baseline + none-mode behavior
amps_all_1, chis_all_1 = of_new.sliding_fit(trace_long, hop=1, reanchor_every=256, chisq_mode='all')
amps_none_1, chis_none_1 = of_new.sliding_fit(trace_long, hop=1, reanchor_every=256, chisq_mode='none')

amps_old_1 = np.empty_like(amps_all_1)
chis_old_1 = np.empty_like(chis_all_1)
for i in range(amps_all_1.size):
    a, c = of_old.fit(trace_long[i:i + N])
    amps_old_1[i] = a
    chis_old_1[i] = c

print('hop=1 all vs old amp maxdiff:', np.max(np.abs(amps_all_1 - amps_old_1)))
print('hop=1 all vs old chi maxdiff:', np.max(np.abs(chis_all_1 - chis_old_1)))
print('hop=1 none chis all NaN:', np.all(np.isnan(chis_none_1)))
print('hop=1 none amps == all amps:', np.allclose(amps_none_1, amps_all_1, atol=1e-12, rtol=1e-10))

assert np.max(np.abs(amps_all_1 - amps_old_1)) < 1e-10
assert np.max(np.abs(chis_all_1 - chis_old_1)) < 1e-18
assert np.all(np.isnan(chis_none_1))
assert np.allclose(amps_none_1, amps_all_1, atol=1e-12, rtol=1e-10)

hop=1 all vs old amp maxdiff: 2.4868995751603507e-14
hop=1 all vs old chi maxdiff: 7.472090494253424e-27
hop=1 none chis all NaN: True
hop=1 none amps == all amps: True


In [5]:
# hop>1: check old baseline + none-mode behavior
hop = int(cfg.get('hop', 16))
if hop <= 1:
    hop = 16
amps_all_h, chis_all_h = of_new.sliding_fit(trace_long, hop=hop, reanchor_every=256, chisq_mode='all')
amps_none_h, chis_none_h = of_new.sliding_fit(trace_long, hop=hop, reanchor_every=256, chisq_mode='none')

starts = np.arange(0, L - N + 1, hop, dtype=int)
amps_old_h = np.empty_like(amps_all_h)
chis_old_h = np.empty_like(chis_all_h)
for j, s in enumerate(starts):
    a, c = of_old.fit(trace_long[s:s + N])
    amps_old_h[j] = a
    chis_old_h[j] = c

print('hop>1 all vs old amp maxdiff:', np.max(np.abs(amps_all_h - amps_old_h)))
print('hop>1 all vs old chi maxdiff:', np.max(np.abs(chis_all_h - chis_old_h)))
print('hop>1 none chis all NaN:', np.all(np.isnan(chis_none_h)))
print('hop>1 none amps == all amps:', np.allclose(amps_none_h, amps_all_h, atol=1e-12, rtol=1e-10))

assert np.max(np.abs(amps_all_h - amps_old_h)) < 1e-10
assert np.max(np.abs(chis_all_h - chis_old_h)) < 1e-18
assert np.all(np.isnan(chis_none_h))
assert np.allclose(amps_none_h, amps_all_h, atol=1e-12, rtol=1e-10)

print('PASS: chisq_mode=\'none\' behavior is fixed for hop=1 and hop>1.')

hop>1 all vs old amp maxdiff: 1.6964207816272392e-13
hop>1 all vs old chi maxdiff: 1.1187940902206478e-25
hop>1 none chis all NaN: True
hop>1 none amps == all amps: True
PASS: chisq_mode='none' behavior is fixed for hop=1 and hop>1.
